# Notebook 02: AgentCore Runtime Setup

## Learning Objectives
- Configure AgentCore Runtime for travel agent
- Create basic conversational agent with Strands
- Deploy the agent and hold a multi-turn conversation

This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


## Step 1: Connect to your AWS environment

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// Remove any existing credential env vars to force profile usage
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

console.log("\u2705 AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { Agent, BedrockModel, tool } from "@strands-agents/sdk";
import { z } from "zod";
import { Runtime } from "../toolkit/mod.ts";
import { loadEnv, state, traceTools, writeFile } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

console.log("\u2705 AgentCore Runtime imports successful");

## Step 2: Create Basic Travel Agent Tools

In [ ]:
// Define basic travel tools for our agent
const getTravelPreferences = tool({
  name: "get_travel_preferences",
  description: "Get user's travel preferences from memory",
  inputSchema: z.object({}),
  // Mock implementation - will be enhanced in Memory notebook
  callback: () => ({
    hotel_type: "mid-range",
    food_preference: "vegetarian",
    budget_range: "moderate",
  }),
});

const calculateBudget = tool({
  name: "calculate_budget",
  description: "Calculate daily budget allocation for travel",
  inputSchema: z.object({
    total_budget: z.number().describe("Total travel budget"),
    days: z.number().describe("Number of days"),
  }),
  // Basic budget breakdown
  callback: ({ total_budget, days }) => ({
    daily_budget: total_budget / days,
    allocation: {
      flights: total_budget * 0.24, // 24%
      hotels: total_budget * 0.36, // 36%
      food: total_budget * 0.20, // 20%
      activities: total_budget * 0.16, // 16%
      buffer: total_budget * 0.04, // 4%
    },
  }),
});

const destinations: Record<string, unknown> = {
  rome: { country: "Italy", currency: "EUR", language: "Italian", attractions: ["Colosseum", "Vatican", "Trevi Fountain"] },
  florence: { country: "Italy", currency: "EUR", language: "Italian", attractions: ["Uffizi Gallery", "Ponte Vecchio", "Duomo"] },
  venice: { country: "Italy", currency: "EUR", language: "Italian", attractions: ["St. Mark's Square", "Grand Canal", "Doge's Palace"] },
};

const getDestinationInfo = tool({
  name: "get_destination_info",
  description: "Get basic information about a travel destination",
  inputSchema: z.object({ destination: z.string().describe("Destination city") }),
  callback: ({ destination }) => destinations[destination.toLowerCase()] ?? { error: "Destination not found" },
});

console.log("\u2705 Travel tools defined");

## Step 3: Create Travel Agent with Bedrock Model

In [ ]:
// Initialize Bedrock model for the agent
const modelId = "us.anthropic.claude-sonnet-4-6";
const model = new BedrockModel({ modelId });

// Create the travel agent with system prompt
const systemPrompt = `
You are an AI Travel Companion specializing in planning trips to Italy.
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
`;

const travelAgent = new Agent({
  model,
  tools: [getTravelPreferences, calculateBudget, getDestinationInfo],
  systemPrompt,
});

// Show each tool call under the cell. Strands' own printer writes to the kernel's stdout,
// which the Deno kernel does not forward to the notebook.
traceTools(travelAgent);

console.log(`\u2705 Travel agent created with ${modelId}`);

## Step 4: Test Local Agent Functionality

In [ ]:
// Test the agent locally before deploying to runtime
async function testTravelAgent(userInput: string): Promise<string> {
  console.log(`User: ${userInput}`);
  const result = await travelAgent.invoke(userInput);
  const agentResponse = result.toString();
  console.log(`Agent: ${agentResponse}`);
  return agentResponse;
}

// Test basic functionality
console.log("\ud83e\uddea Testing Travel Agent Locally");
console.log("=".repeat(50));

await testTravelAgent("Hi, I want to plan a trip to Italy");

In [ ]:
// Test budget calculation
await testTravelAgent("I have a budget of $5000 for a 10-day trip. How should I allocate it?");

In [ ]:
// Test destination information
await testTravelAgent("Tell me about Rome and what I should see there");

## Step 5: Prepare Agent for AgentCore Runtime

The two cells below write the deployable agent folder, replacing the `%%writefile` cells of the
Python course: the agent's entry file and its `deno.json`, which plays the role `requirements.txt`
played before.

In [ ]:
await writeFile("../backend/runtime/simple_agent/travel_agent.ts", `import { Agent, BedrockModel, tool } from "@strands-agents/sdk";
import { BedrockAgentCoreApp } from "bedrock-agentcore/runtime";
import { z } from "zod";

// Define travel tools
const getTravelPreferences = tool({
  name: "get_travel_preferences",
  description: "Get user's travel preferences from memory",
  inputSchema: z.object({}),
  callback: () => ({
    hotel_type: "mid-range",
    food_preference: "vegetarian",
    budget_range: "moderate",
  }),
});

const calculateBudget = tool({
  name: "calculate_budget",
  description: "Calculate daily budget allocation for travel",
  inputSchema: z.object({
    total_budget: z.number().describe("Total travel budget"),
    days: z.number().describe("Number of days"),
  }),
  callback: ({ total_budget, days }) => ({
    daily_budget: total_budget / days,
    allocation: {
      flights: total_budget * 0.24,
      hotels: total_budget * 0.36,
      food: total_budget * 0.20,
      activities: total_budget * 0.16,
      buffer: total_budget * 0.04,
    },
  }),
});

const destinations: Record<string, unknown> = {
  rome: {
    country: "Italy",
    currency: "EUR",
    language: "Italian",
    attractions: ["Colosseum", "Vatican", "Trevi Fountain"],
  },
  florence: {
    country: "Italy",
    currency: "EUR",
    language: "Italian",
    attractions: ["Uffizi Gallery", "Ponte Vecchio", "Duomo"],
  },
  venice: {
    country: "Italy",
    currency: "EUR",
    language: "Italian",
    attractions: ["St. Mark's Square", "Grand Canal", "Doge's Palace"],
  },
};

const getDestinationInfo = tool({
  name: "get_destination_info",
  description: "Get basic information about a travel destination",
  inputSchema: z.object({ destination: z.string().describe("Destination city") }),
  callback: ({ destination }) =>
    destinations[destination.toLowerCase()] ?? { error: "Destination not found" },
});

// Initialize model and agent
const modelId = "us.anthropic.claude-sonnet-4-6";
const model = new BedrockModel({ modelId });

const systemPrompt = \`
You are an AI Travel Companion specializing in planning trips to Italy.
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
\`;

const travelAgent = new Agent({
  model,
  tools: [getTravelPreferences, calculateBudget, getDestinationInfo],
  systemPrompt,
});

// AgentCore Runtime entrypoint for travel agent
const app = new BedrockAgentCoreApp({
  invocationHandler: {
    requestSchema: z.object({ prompt: z.string().default("") }),
    process: async ({ prompt }) => {
      console.log(\`User input: \${prompt}\`);
      const result = await travelAgent.invoke(prompt);
      return result.toString();
    },
  },
});

app.run();
`);

In [ ]:
await writeFile("../backend/runtime/simple_agent/deno.json", `{
  "nodeModulesDir": "auto",
  "compilerOptions": {
    "strict": true
  },
  "imports": {
    "@strands-agents/sdk": "npm:@strands-agents/sdk@1.18.0",
    "bedrock-agentcore/": "npm:/bedrock-agentcore@0.4.4/",
    "zod": "npm:zod@4.6.5",
    "@opentelemetry/api": "npm:@opentelemetry/api@1.9.1",
    "@opentelemetry/api-logs": "npm:@opentelemetry/api-logs@0.219.0",
    "@opentelemetry/context-async-hooks": "npm:@opentelemetry/context-async-hooks@2.8.0",
    "@opentelemetry/core": "npm:@opentelemetry/core@2.8.0",
    "@opentelemetry/otlp-transformer": "npm:@opentelemetry/otlp-transformer@0.219.0",
    "@opentelemetry/resources": "npm:@opentelemetry/resources@2.8.0",
    "@opentelemetry/sdk-logs": "npm:@opentelemetry/sdk-logs@0.219.0",
    "@opentelemetry/sdk-trace-base": "npm:@opentelemetry/sdk-trace-base@2.8.0",
    "@smithy/protocol-http": "npm:@smithy/protocol-http@5.6.2",
    "@smithy/signature-v4": "npm:@smithy/signature-v4@5.7.3",
    "@aws-crypto/sha256-js": "npm:@aws-crypto/sha256-js@5.2.0",
    "@aws-sdk/credential-provider-node": "npm:@aws-sdk/credential-provider-node@3.972.83"
  }
}
`);

## Step 6: Deploy to AgentCore Runtime

`Runtime` builds the container remotely in CodeBuild, so no local Docker is needed. Unlike the
Python version it takes `sourceDir`, so the notebook does not change working directory.

In [ ]:
// Initialize runtime deployment
const region = Deno.env.get("AWS_REGION") ?? "us-east-1";
const agentName = "travel_companion_basic";
const sourceDir = "../backend/runtime/simple_agent";

const agentcoreRuntime = new Runtime();

console.log("\ud83d\ude80 Configuring AgentCore Runtime deployment...");
console.log(`Agent Name: ${agentName}`);
console.log(`Region: ${region}`);
console.log(`Source: ${sourceDir}`);

const configureResponse = await agentcoreRuntime.configure({
  entrypoint: "travel_agent.ts",
  autoCreateExecutionRole: true,
  autoCreateEcr: true,
  requirementsFile: "deno.json",
  region,
  agentName,
  sourceDir,
});

console.log("\u2705 Configured:", configureResponse);

In [ ]:
// Launch the agent to AgentCore Runtime
console.log("\ud83d\ude80 Launching agent to AgentCore Runtime...");
console.log("This may take 5-10 minutes...");

const launchResult = await agentcoreRuntime.launch();
console.log("\u2705 Launch complete");
console.log(`Agent ARN: ${launchResult.agentArn}`);
console.log(`ECR URI: ${launchResult.ecrUri}`);

In [ ]:
// Check deployment status
console.log("\u23f3 Checking deployment status...");
let statusResponse = await agentcoreRuntime.status();
let status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
const endStatus = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"];

while (!endStatus.includes(status)) {
  console.log(`Status: ${status}`);
  await new Promise((resolve) => setTimeout(resolve, 30_000)); // Check every 30 seconds
  statusResponse = await agentcoreRuntime.status();
  status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
}

console.log(`\n\ud83c\udf89 Final Status: ${status}`);
console.log(
  status === "READY"
    ? "\u2705 Agent successfully deployed to AgentCore Runtime!"
    : "\u274c Deployment failed. Check AWS console for details.",
);

## Step 7: Test Deployed Agent

In [ ]:
// Test the deployed agent
if (status === "READY") {
  console.log("\ud83e\uddea Testing deployed Travel Agent");
  console.log("=".repeat(50));

  // Test basic interaction
  const testPayload = { prompt: "Hi, I want to plan a 10-day trip to Italy with a $5000 budget" };
  const invokeResponse = await agentcoreRuntime.invoke(testPayload);

  console.log(`User: ${testPayload.prompt}`);
  console.log(`Agent: ${invokeResponse}`);
} else {
  console.log("\u26a0\ufe0f Cannot test - deployment not ready");
}

In [ ]:
// Test budget calculation functionality
if (status === "READY") {
  const testPayload = { prompt: "Can you help me allocate my $5000 budget for 10 days?" };
  const invokeResponse = await agentcoreRuntime.invoke(testPayload);

  console.log(`\nUser: ${testPayload.prompt}`);
  console.log(`Agent: ${invokeResponse}`);
}

## Step 8: Multi-turn Conversation Test

In [ ]:
// Test multi-turn conversation
if (status === "READY") {
  console.log("\ud83d\udde3\ufe0f Testing Multi-turn Conversation");
  console.log("=".repeat(50));

  const conversation = [
    "I want to visit Italy",
    "I prefer mid-range hotels and vegetarian food",
    "Tell me about Rome's attractions",
    "What about Florence?",
  ];

  for (const [index, message] of conversation.entries()) {
    console.log(`\n--- Turn ${index + 1} ---`);
    const invokeResponse = await agentcoreRuntime.invoke({ prompt: message });

    console.log(`User: ${message}`);
    console.log(`Agent: ${String(invokeResponse).slice(0, 200)}...`); // Truncate for readability
  }
}

## Step 9: Save Runtime Information

In [ ]:
// Save runtime information for use in subsequent notebooks
if (status === "READY") {
  const runtimeInfo = {
    agent_name: agentName,
    agent_arn: launchResult.agentArn,
    agent_id: launchResult.agentId,
    ecr_uri: launchResult.ecrUri,
    region,
    status,
  };

  // Save to file for next notebooks
  await writeFile("environments/runtime_info.json", `${JSON.stringify(runtimeInfo, null, 2)}\n`);
  // Also keep it in the notebook state, so later notebooks can read it after a kernel restart
  await state.set("runtime_info", runtimeInfo);

  console.log("\ud83d\udcbe Runtime information saved to environments/runtime_info.json");
  console.log("\n\ud83d\udccb Runtime Summary:");
  for (const [key, value] of Object.entries(runtimeInfo)) {
    console.log(`  ${key}: ${value}`);
  }
} else {
  console.log("\u26a0\ufe0f Runtime information not saved - deployment not ready");
}

## Next Steps

✅ **Completed in this notebook:**
- AgentCore Runtime configuration and setup
- Basic travel agent with Strands tools
- Deployment to AgentCore Runtime via CodeBuild
- Multi-turn conversation testing

➡️ **Next: Notebook 03 - Gateway Integration**
- Connect the agent to external APIs through AgentCore Gateway
